In [8]:
import pandas as pd
import numpy as np

# 剧毒原始数据：融合了 DataCamp 全章节的脏乱差
df_users = pd.DataFrame({
    'user_id': ['U01', 'U 002', 'U@03', 'U  01', 'U0400000', 'UA134', 'U3786A'], # 注意：有重复数据
    'birth_year': ['1990', '1985', '2050', '1990', '1992', np.nan, '1988'], # 注意：有未来年份、缺失值
    'age': [36, 41, -5, 36, 33, 25, 38], # 假设当前是 2026 年
    'income': ['$5,000', '6000.50', 'NaN', '$5,000', '-1000', ' 8000 ', '$$9000'], # 字符串混杂符号、非法负数
    'company': ['Tencent', ' Alibaba ', 'ByteDance', 'Tencent', 'Alibba', 'McDonalds', 'Apple Inc']
})

print("🚨 原始剧毒数据集：")
print(df_users.info())

🚨 原始剧毒数据集：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     7 non-null      object
 1   birth_year  6 non-null      object
 2   age         7 non-null      int64 
 3   income      7 non-null      object
 4   company     7 non-null      object
dtypes: int64(1), object(4)
memory usage: 412.0+ bytes
None


In [9]:
# back up raw data
df_clean = df_users.copy()

In [10]:
# cleaning user_id

# anomalous-length boolean sequence and raw data 
is_anomalous_length = (df_clean['user_id'].str.len() < 2) | (df_clean['user_id'].str.len() > 6)
anomalous_length_data = df_clean[is_anomalous_length]
normal_length_data = df_clean[~is_anomalous_length]
print(anomalous_length_data['user_id'].str.len().value_counts().head())

# define data standard patterns via regex
standard_pattern_id = r'^U\d{2,5}$'
is_nonstandard_pattern_id = ~normal_length_data['user_id'].str.match(standard_pattern_id,na=False)
nonstandard_pattern_id = normal_length_data[is_nonstandard_pattern_id]
print(nonstandard_pattern_id.head())


# ==========================================
# 动作 1：纯粹的清洗引擎 (只管洗，绝对不删减行数！)
# ==========================================
def clean_id_column(raw_id_series):
    # 暴力提取纯数字并加上 U
    pure_digits = raw_id_series.str.replace(r'\D', '', regex=True)
    cleaned_id = 'U' + pure_digits
    
    # 终极掩码与分流
    standard_pattern = r'^U\d{2,5}$'
    valid_mask = cleaned_id.str.match(standard_pattern, na=False)
    
    # 进多少行，出多少行，不多不少
    return np.where(valid_mask, cleaned_id, 'Unknown')

# ==========================================
# 动作 2：表级流水线控制 (在这里删减、去重)
# ==========================================
# 1. 执行原位清洗 (长度绝对一致，完美塞入)
df_clean['user_id'] = clean_id_column(df_clean['user_id'])

# 2. 拿到全表之后，开始分离废料
df_unknown = df_clean[df_clean['user_id'] == 'Unknown']
df_valid = df_clean[df_clean['user_id'] != 'Unknown']

# 3. 对合法全表进行去重
df_valid = df_valid.drop_duplicates(subset=['user_id'], keep='last')

# 4. 合流
df_final = pd.concat([df_valid, df_unknown])

print(df_final)

    




user_id
8    1
Name: count, dtype: int64
  user_id birth_year  age   income    company
1   U 002       1985   41  6000.50   Alibaba 
2    U@03       2050   -5      NaN  ByteDance
3   U  01       1990   36   $5,000    Tencent
5   UA134        NaN   25    8000   McDonalds
6  U3786A       1988   38   $$9000  Apple Inc
   user_id birth_year  age   income    company
1     U002       1985   41  6000.50   Alibaba 
2      U03       2050   -5      NaN  ByteDance
3      U01       1990   36   $5,000    Tencent
5     U134        NaN   25    8000   McDonalds
6    U3786       1988   38   $$9000  Apple Inc
4  Unknown       1992   33    -1000     Alibba
